# Importo le cartelle e le repository

In [3]:
import os
import sys
import glob
import random
import time
import numpy as np

import torch
from torch.utils.data import DataLoader

from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from sklearn.metrics import average_precision_score, roc_curve

print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


NumPy: 1.26.4
Torch: 2.10.0+cu128
CUDA available: True


In [4]:
# collego colab al drive
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [6]:
BASE_PATH = '/content/drive/MyDrive/Validation_Dataset'
CITYSCAPES_DRIVE_PATH = '/content/drive/MyDrive/validation_Cityscapes'

print("Validation Dataset exists:", os.path.exists(BASE_PATH))
if os.path.exists(BASE_PATH):
    print(os.listdir(BASE_PATH))

print("\nvalidation_Cityscapes exists:", os.path.exists(CITYSCAPES_DRIVE_PATH))
if os.path.exists(CITYSCAPES_DRIVE_PATH):
    print(os.listdir(CITYSCAPES_DRIVE_PATH))

Validation Dataset exists: True
['.DS_Store', 'RoadAnomaly21', 'RoadObsticle21', 'fs_static', 'FS_LostFound_full', 'RoadAnomaly']

validation_Cityscapes exists: True
['gtFine_trainvaltest.zip', 'eomt_cityscapes.bin', 'eomt_coco.bin', 'leftImg8bit_trainvaltest.zip', 'license.txt', 'README', 'Validation_Dataset']


In [8]:
#Clonare la repository GitHub con il codice
# !git clone https://github.com/GiammaBigFishEngineer/MaskArchitectureAnomaly_CourseProject.git
!git clone https://github.com/elenic9/MaskArchitectureAnomaly_CourseProject.git

#entrare dentro la cartella "eval" della repository
%cd MaskArchitectureAnomaly_CourseProject/eval

import sys
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eval')

Cloning into 'MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 131, done.
remote: Total 131 (delta 0), reused 0 (delta 0), pack-reused 131 (from 1)
Receiving objects: 100% (131/131), 26.87 MiB | 21.88 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/MaskArchitectureAnomaly_CourseProject/eval


# Inizializzazione del modello

importo il modello e carico i pesi

In [9]:
from erfnet import ERFNet

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
NUM_CLASSES = 20
weightspath = '/content/MaskArchitectureAnomaly_CourseProject/trained_models/erfnet_pretrained.pth'

model = ERFNet(NUM_CLASSES).cuda()

def load_my_state_dict(model, state_dict):
    own_state = model.state_dict()
    for name, param in state_dict.items():
        if name not in own_state:
            if name.startswith("module."):
                own_state[name.split("module.")[-1]].copy_(param)
            else:
                print(name, " not loaded")
                continue
        else:
            own_state[name].copy_(param)
    return model

model = load_my_state_dict(model, torch.load(weightspath, map_location=lambda storage, loc: storage))
print ("Model and weights LOADED successfully")
model.eval()

Model and weights LOADED successfully


ERFNet(
  (encoder): Encoder(
    (initial_block): DownsamplerBlock(
      (conv): Conv2d(3, 13, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    )
    (layers): ModuleList(
      (0): DownsamplerBlock(
        (conv): Conv2d(16, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1-5): 5 x non_bottleneck_1d(
        (conv3x1_1): Conv2d(64, 64, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
        (conv1x3_1): Conv2d(64, 64, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
        (bn1): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (conv3x1_2): Conv2d(64

trasformazioni geometriche e di formato per allineare i dati alla rete.
copiato dal file evalAnomaly.py


In [10]:
from PIL import Image


input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        ToTensor(),
        # Normalize([.485, .456, .406], [.229, .224, .225]),
    ]
)

target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)

# Inferenza e calcolo anomaly scores

In [11]:
!pip install ood_metrics
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr,plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score

In [12]:

# Dataset da valutare
datasets = {
    "SMIYC RA-21": "RoadAnomaly21",
    "SMIYC RO-21": "RoadObsticle21",
    "FS L&F": "FS_LostFound_full",
    "FS Static": "fs_static",
    "Road Anomaly": "RoadAnomaly"
}

# Dizionario finale in cui salvo i risultati
all_results = {}

for dataset_name, dataset_folder in datasets.items():

    print("\n" + "=" * 80)
    print(f"DATASET: {dataset_name} ({dataset_folder})")
    print("=" * 80)

    # Liste reinizializzate per ogni dataset
    ood_gts_list = []
    anomaly_scores_dict = {
        'max_logit': [],
        'msp': [],
        'entropy': []
    }

    input_path = f'{BASE_PATH}/{dataset_folder}/images/*'
    image_paths = sorted(glob.glob(input_path))

    print(f"Numero immagini trovate: {len(image_paths)}")

    for idx, path in enumerate(image_paths):

        if idx % 10 == 0:
            print(f"  Processing image {idx+1}/{len(image_paths)}")

        images = input_transform(Image.open(path).convert('RGB')).unsqueeze(0).float().cuda()

        with torch.no_grad():
            result = model(images)

        # -------------------------
        # Calcolo anomaly scores
        # -------------------------

        # Metodo 1: MaxLogit
        score_maxLogit = 1.0 - np.max(result.squeeze(0).data.cpu().numpy(), axis=0)

        # Metodo 2: MSP
        probs_tensor = torch.softmax(result, dim=1)
        probs_np = probs_tensor.squeeze(0).data.cpu().numpy()
        score_msp = 1.0 - np.max(probs_np, axis=0)

        # Metodo 3: Max Entropy
        entropy_tensor = torch.sum(
            -probs_tensor * torch.log(probs_tensor + 1e-10),
            dim=1
        ) / torch.log(torch.tensor(NUM_CLASSES, dtype=torch.float32).cuda())

        score_entropy = entropy_tensor.squeeze(0).data.cpu().numpy()

        # -------------------------
        # Lettura ground truth
        # -------------------------

        pathGT = path.replace("images", "labels_masks")

        if dataset_folder == "RoadObsticle21":
            pathGT = pathGT.replace("webp", "png")

        if dataset_folder in ["fs_static", "RoadAnomaly", "RoadAnomaly21", "FS_LostFound_full"]:
            pathGT = pathGT.replace("jpg", "png")

        mask = Image.open(pathGT)
        mask = target_transform(mask)
        ood_gts = np.array(mask)

        # -------------------------
        # Conversione labels GT
        # -------------------------
        if dataset_folder in ["RoadAnomaly", "RoadAnomaly21"]:
            ood_gts = np.where(ood_gts == 2, 1, ood_gts)

        # Filtro protettivo per immagini valide
        if 1 not in np.unique(ood_gts):
            continue

        anomaly_scores_dict['max_logit'].append(score_maxLogit)
        anomaly_scores_dict['msp'].append(score_msp)
        anomaly_scores_dict['entropy'].append(score_entropy)
        ood_gts_list.append(ood_gts)

        del result, score_maxLogit, score_msp, score_entropy
        del ood_gts, mask, probs_tensor, probs_np, entropy_tensor, images
        torch.cuda.empty_cache()

    # -------------------------
    # Concatenazione per dataset
    # -------------------------

    if len(ood_gts_list) == 0:
        print(f"Nessuna immagine valida trovata per {dataset_name}")
        continue

    ood_gts_all = np.concatenate(ood_gts_list)
    anomaly_scores_maxLogit = np.concatenate(anomaly_scores_dict['max_logit'])
    anomaly_scores_msp = np.concatenate(anomaly_scores_dict['msp'])
    anomaly_scores_entropy = np.concatenate(anomaly_scores_dict['entropy'])

    # Salvo temporaneamente per il calcolo metriche
    all_results[dataset_name] = {
        "gt": ood_gts_all,
        "max_logit": anomaly_scores_maxLogit,
        "msp": anomaly_scores_msp,
        "entropy": anomaly_scores_entropy
    }

    print(f"Immagini valide usate: {len(ood_gts_list)}")
    print("\nDataset effettivamente salvati in all_results:")

print(list(all_results.keys()))


DATASET: SMIYC RA-21 (RoadAnomaly21)
Numero immagini trovate: 10
  Processing image 1/10
Immagini valide usate: 10

Dataset effettivamente salvati in all_results:

DATASET: SMIYC RO-21 (RoadObsticle21)
Numero immagini trovate: 30
  Processing image 1/30
  Processing image 11/30
  Processing image 21/30
Immagini valide usate: 30

Dataset effettivamente salvati in all_results:

DATASET: FS L&F (FS_LostFound_full)
Numero immagini trovate: 100
  Processing image 1/100
  Processing image 11/100
  Processing image 21/100
  Processing image 31/100
  Processing image 41/100
  Processing image 51/100
  Processing image 61/100
  Processing image 71/100
  Processing image 81/100
  Processing image 91/100
Immagini valide usate: 99

Dataset effettivamente salvati in all_results:

DATASET: FS Static (fs_static)
Numero immagini trovate: 30
  Processing image 1/30
  Processing image 11/30
  Processing image 21/30
Immagini valide usate: 20

Dataset effettivamente salvati in all_results:

DATASET: Road

In [13]:
final_table = {}

for dataset_name, data in all_results.items():

    print("\n" + "=" * 80)
    print(f"RISULTATI DATASET: {dataset_name}")
    print("=" * 80)

    ood_gts_all = data["gt"]

    ood_mask = (ood_gts_all == 1)
    ind_mask = (ood_gts_all == 0)

    concatenated_scores = {
        'MSP': data["msp"],
        'MaxLogit': data["max_logit"],
        'MaxEntropy': data["entropy"]
    }

    final_table[dataset_name] = {}

    for method_name, current_score in concatenated_scores.items():

        # Pixel OOD e ID
        ood_out = current_score[ood_mask]
        ind_out = current_score[ind_mask]

        ood_label = np.ones(len(ood_out))
        ind_label = np.zeros(len(ind_out))

        val_out = np.concatenate((ind_out, ood_out))
        val_label = np.concatenate((ind_label, ood_label))

        # Metriche
        auprc = average_precision_score(val_label, val_out)
        fpr = fpr_at_95_tpr(val_out, val_label)

        final_table[dataset_name][method_name] = {
            "AUPRC": auprc * 100,
            "FPR95": fpr * 100
        }

        print(f"{method_name}")
        print(f"  AUPRC: {auprc * 100:.2f}%")
        print(f"  FPR95: {fpr * 100:.2f}%")
        print("-" * 40)


RISULTATI DATASET: SMIYC RA-21
MSP
  AUPRC: 29.10%
  FPR95: 62.55%
----------------------------------------
MaxLogit
  AUPRC: 38.32%
  FPR95: 59.34%
----------------------------------------
MaxEntropy
  AUPRC: 30.97%
  FPR95: 62.66%
----------------------------------------

RISULTATI DATASET: SMIYC RO-21
MSP
  AUPRC: 2.71%
  FPR95: 65.22%
----------------------------------------
MaxLogit
  AUPRC: 4.63%
  FPR95: 48.44%
----------------------------------------
MaxEntropy
  AUPRC: 3.04%
  FPR95: 65.91%
----------------------------------------

RISULTATI DATASET: FS L&F
MSP
  AUPRC: 1.75%
  FPR95: 50.59%
----------------------------------------
MaxLogit
  AUPRC: 3.30%
  FPR95: 45.49%
----------------------------------------
MaxEntropy
  AUPRC: 2.58%
  FPR95: 50.16%
----------------------------------------

RISULTATI DATASET: FS Static
MSP
  AUPRC: 7.47%
  FPR95: 41.84%
----------------------------------------
MaxLogit
  AUPRC: 9.50%
  FPR95: 40.30%
----------------------------------------

# mIoU

Preparazione Cityscape

In [14]:
!pip install -q cityscapesscripts

!rm -rf /content/cityscapes_data
!mkdir -p /content/cityscapes_data

!unzip -q /content/drive/MyDrive/validation_Cityscapes/leftImg8bit_trainvaltest.zip "leftImg8bit/val/*" -d /content/cityscapes_data/
!unzip -q /content/drive/MyDrive/validation_Cityscapes/gtFine_trainvaltest.zip "gtFine/val/*" -d /content/cityscapes_data/

# Crea i file *_labelTrainIds.png se non sono già presenti.
%env CITYSCAPES_DATASET=/content/cityscapes_data
!python -m cityscapesscripts.preparation.createTrainIdLabelImgs

base = "/content/cityscapes_data"
print("Immagini val:", len(glob.glob(base + "/leftImg8bit/val/*/*leftImg8bit.png")))
print("GT labelTrainIds:", len(glob.glob(base + "/gtFine/val/*/*gtFine_labelTrainIds.png")))
print(glob.glob(base + "/gtFine/val/*/*gtFine_labelTrainIds.png")[:3])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.6/473.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.8 MB/s eta 0:00:00
env: CITYSCAPES_DATASET=/content/cityscapes_data
Processing 500 annotation files
Progress: 100.0 % Immagini val: 500
GT labelTrainIds: 500
['/content/cityscapes_data/gtFine/val/munster/munster_000018_000019_gtFine_labelTrainIds.png', '/content/cityscapes_data/gtFine/val/munster/munster_000032_000019_gtFine_labelTrainIds.png', '/content/cityscapes_data/gtFine/val/munster/munster_000065_000019_gtFine_labelTrainIds.png']


Calcolo mIoU

In [15]:

from dataset import cityscapes
from transform import Relabel, ToLabel, Colorize
from iouEval import iouEval, getColorEntry

# --- CONFIGURAZIONE VARIABILI (Al posto di ArgumentParser) ---
class Args:
    state = None
    loadDir = "../trained_models/"                 # Cartella dei pesi
    loadWeights = "erfnet_pretrained.pth"       # Nome del file pesi
    loadModel = "erfnet.py"
    subset = "val"                               # 'val' o 'train'
    datadir = "/content/cityscapes_data/" # Percorso dati su Drive
    num_workers = 2
    batch_size = 1
    cpu = False                                  # Metti False per usare la GPU di Colab

args = Args()


NUM_CLASSES = 20


input_transform_cityscapes = Compose([
    Resize(512, Image.BILINEAR),
    ToTensor(),
])
target_transform_cityscapes = Compose([
    Resize(512, Image.NEAREST),
    ToLabel(),
    Relabel(255, 19),   #ignore label to 19
])

weightspath = args.loadDir + args.loadWeights
print ("Loading weights: " + weightspath)

model_miou = ERFNet(NUM_CLASSES)

if (not args.cpu):
    model = torch.nn.DataParallel(model_miou).cuda()

state_dict = torch.load(weightspath, map_location=lambda storage, loc: storage)
model_miou = load_my_state_dict(model_miou, state_dict)
model_miou.eval()

if(not os.path.exists(args.datadir)):
    print ("Error: datadir could not be loaded")

loader = DataLoader(
    cityscapes(args.datadir, input_transform_cityscapes, target_transform_cityscapes, subset=args.subset),
    num_workers=args.num_workers,
    batch_size=args.batch_size,
    shuffle=False
)

iouEvalVal = iouEval(NUM_CLASSES)
start = time.time()

for step, (images, labels, filename, filenameGt) in enumerate(loader):
    if (not args.cpu):
        images = images.cuda()
        labels = labels.cuda()

    with torch.no_grad():
        outputs = model_miou(images)

    iouEvalVal.addBatch(outputs.max(1)[1].unsqueeze(1).data, labels)

    #filenameSave = filename[0].split("leftImg8bit/")[1]

    #print (step, filenameSave)

iouVal, iou_classes = iouEvalVal.getIoU()

class_names = [
    "Road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

print("---------------------------------------")
print("Took ", time.time()-start, "seconds")
print("=======================================")
print("Per-Class IoU:")
for i, name in enumerate(class_names):
    print(f"{name}: {iou_classes[i] * 100:.2f}%")

print("=======================================")
print(f"MEAN IoU: {iouVal * 100:.2f}%")


Loading weights: ../trained_models/erfnet_pretrained.pth
/content/cityscapes_data/leftImg8bit/val /content/cityscapes_data/gtFine/val
---------------------------------------
Took  78.75962042808533 seconds
Per-Class IoU:
Road: 97.62%
sidewalk: 81.37%
building: 90.77%
wall: 49.43%
fence: 54.93%
pole: 60.81%
traffic light: 62.60%
traffic sign: 72.32%
vegetation: 91.35%
terrain: 60.97%
sky: 93.38%
person: 76.11%
rider: 53.45%
car: 92.91%
truck: 72.78%
bus: 78.87%
train: 63.86%
motorcycle: 46.41%
bicycle: 71.89%
MEAN IoU: 72.20%
